# AI Mind Reader — evaluation walkthrough

This notebook reads the artifacts produced by the scripts; it does not
recompute anything, so the numbers here and in the README can never
disagree.

Regenerate the inputs first:

```bash
python scripts/build_dataset.py
python scripts/train_models.py
python scripts/evaluate.py
```

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"

from mindreader.dataset import load_prototypes

protos = load_prototypes()
print(f"{protos.n_objects} objects x {protos.n_features} attributes")

## 1. The dataset

Compiled from `data/source/*.json`, never hand-edited. The build refuses to
emit a dataset where two objects are separated by fewer than three answers —
if they differ by one, a single player mistake makes them indistinguishable.

In [ ]:
import itertools

import numpy as np

X = protos.X
d = np.array([
    (X[i] != X[j]).sum()
    for i, j in itertools.combinations(range(protos.n_objects), 2)
])
print(f"pairwise distance: min={d.min()} median={np.median(d):.0f} max={d.max()}")
print(f"pairs 1 answer apart: {(d == 1).sum()}   2 apart: {(d == 2).sum()}")

pd.Series(protos.categories).value_counts().plot.barh(
    title="Objects per category", figsize=(6, 3.5)
);

## 2. Model comparison (Phase 3)

117-class object identification, scored against simulated players who get
some answers wrong. `BernoulliNB` wins; the neural network does not.

In [ ]:
models = pd.read_csv(DATA / "model_comparison.csv")
display(models)

bnb = models.loc[models.Model == "bernoulli_nb", "Noisy Top-1 (%)"].iloc[0]
gnb = models.loc[models.Model == "gaussian_nb", "Noisy Top-1 (%)"].iloc[0]
mlp = models.loc[models.Model == "mlp_neural_network", "Noisy Top-1 (%)"].iloc[0]
print(f"BernoulliNB beats GaussianNB by {bnb - gnb:+.1f} points")
print(f"BernoulliNB beats the MLP by     {bnb - mlp:+.1f} points")

Binary attributes need a Bernoulli likelihood. A Gaussian one models a
yes/no answer as a continuous normal variable, which is simply the wrong
distribution — and it costs about 7 points.

The neural network losing is worth reporting rather than hiding. With binary
attributes and independent answer noise, the Bayes posterior is already close
to optimal, so a 256×128 MLP has nothing left to exploit.

## 3. Whole-game accuracy (Phase 5)

Model accuracy on single answer sheets is not what a player experiences.
This is complete games: the engine choosing its own questions until it
commits to a guess.

In [ ]:
ev = pd.read_csv(DATA / "evaluation.csv")
display(ev)

ax = ev.plot(
    x="player_error_rate", y=["top1_pct", "top3_pct"],
    marker="o", figsize=(7, 4), ylim=(0, 102),
    title="Whole-game accuracy vs how careless the player is",
)
ax.set_xlabel("share of answers the player gets wrong")
ax.set_ylabel("accuracy (%)")
ax.grid(alpha=0.3)

## 4. Calibration — does the confidence figure mean anything?

Accuracy says how often the guess is right. Calibration asks something
different: when the engine claims 92%, is it right 92% of the time? A
confidence number that fails this is decoration.

Measured over every step of every game, not just final guesses — final
confidences cluster near the stopping threshold and would leave the low end
of the range untested.

In [ ]:
summary = json.loads((DATA / "evaluation_summary.json").read_text())
print(f"expected calibration error: {summary['ece_before']:.3f}")
print(f"confidence minus accuracy:  {summary['overconfidence_before']:+.3f}")
print(f"fitted likelihood weight:   {summary['fitted_likelihood_weight']}")

cal = pd.read_csv(DATA / "calibration.csv")
base = cal[cal.likelihood_weight == 1.0]
ax = base.plot.scatter(
    x="mean_confidence", y="accuracy", s=base["count"] / 40,
    figsize=(5.5, 5.5), title="Reliability diagram (point size = sample count)",
)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("confidence claimed")
ax.set_ylabel("actually right")
ax.grid(alpha=0.3)

The points sit on the diagonal, so the confidence figure is trustworthy.

**I expected the opposite.** Naive Bayes is overconfident when its inputs are
dependent, and these provably are — the dataset enforces 65 implication rules
such as `is_mammal → is_animal`, so answering both appears to count one fact
twice. I added a `likelihood_weight` knob to discount each answer and fitted
it; the best value came out at **1.0**, meaning no correction was needed.

My first explanation was that information gain never asks a redundant
question. That was testable, so I tested it, and it was **wrong**:
implication-linked attributes are asked together *more* often than unrelated
pairs, not less.

The real reason is that this is not really *naive* Bayes. Naive Bayes
approximates `P(features | class)` as a product. Here that product is not an
approximation — it is exactly the simulated noise process, since answers are
flipped independently at a known rate. And with only 117 objects the engine
enumerates the entire hypothesis space rather than assuming anything about
it. The implication rules constrain *which prototype vectors exist*, not *how
answers get corrupted*, so there is no independence assumption left to
violate.

Which makes the result partly circular, and that is worth stating plainly: it
confirms the arithmetic is right, not that the noise model is realistic.
Against real players — reliably sure about `is_animal`, reliably vague about
`is_big` — the model would be misspecified and calibration should degrade.
`data/feedback_log.csv` is what would eventually measure that.

## 5. Where it still fails

The objects the engine gets wrong most often, and what it confuses them with.

In [ ]:
per_obj = pd.read_csv(DATA / "per_object.csv").sort_values("accuracy_pct")
display(per_obj.head(12))

ax = per_obj.head(12).plot.barh(
    x="object", y="accuracy_pct", figsize=(7, 4.5), legend=False,
    title="Hardest objects to guess",
)
ax.set_xlabel("accuracy (%)")

These are the objects whose nearest neighbours are closest in attribute
space — `Scissors` against `Screwdriver`, `Elephant` against `Horse`. The
separability gate guarantees at least three answers between any two objects,
but three is the minimum and it is these pairs that sit on it. Raising the
gate to four would be the direct fix, and it would need new attributes.

## 6. Does the assumed error rate matter?

The engine assumes players answer wrongly 10% of the time. You never
actually know that, so it is worth checking how much the assumption matters.

In [ ]:
sens = pd.DataFrame(
    summary["noise_sensitivity"],
).T
sens.columns = [f"true {int(float(c) * 100)}%" for c in
                [0.0, 0.05, 0.10, 0.15, 0.20]]
sens.index = [f"assumes {int(float(i) * 100)}%" for i in sens.index]
display(sens.style.format("{:.1f}%").background_gradient(cmap="RdYlGn"))

Accuracy is mostly flat across the rows, so the engine is not sensitive to
getting its noise assumption wrong — which matters, because in deployment it
will always be wrong to some degree.